# Investigation sur les résultats de la nouvelle modale de choix de la cc pour les contributions - parcours SEO

Contexte

* Problème de référencement où la version personnalisée de certaines contributions est mieux référencée que la version générique.
* De fait les usagers arrivent sur une contribution déjà personnalisée d'après une cc. Or, l'essentiel des usagers ne faisaient pas l'effort de modifier la cc indiquée (98% de taux de sortie). Ce qui pose 2 problèmes : soit l'usager prend pour acquis une réponse qui ne s'applique pas à sa situation (car ce n'est pas la bonne cc) ; soit il repart tout de suite car il ne comprend pas pourquoi on lui parle de cette cc.
* Nous avons mis en place fin juillet un nouveau design pour davantage inciter les usagers à choisir LEUR propre convention collective.

Issue : https://github.com/SocialGouv/code-du-travail-numerique/issues/7467

In [ ]:
%load_ext dotenv
%dotenv

In [ ]:
import pandas as pd
from analysis.connectors.matomo import MatomoSQLConnector

matomo = MatomoSQLConnector()
await matomo.connect()

In [ ]:
# - 10 juin / now
interval_start = '2026-08-01 00:00:00'
interval_stop = '2026-09-01 00:00:00'

In [ ]:
columns = ['action_id',
    'idvisit',
    'actions',
    'visitduration', 
    'operatingsystemname',
    'action_timestamp',
    'action_type',
    'action_eventcategory',
    'action_eventaction',
    'action_eventname',
    'action_eventvalue',
    'action_url',
    'referrertype', 
    'referrername',
    'experiments']

range_query = f"""
        SELECT {", ".join(columns)} FROM matomo_partitioned
        WHERE action_timestamp >= '{interval_start}'
          AND action_timestamp < '{interval_stop}'
        """

In [ ]:
query_visits =  range_query + f"""
          ORDER BY action_timestamp asc;
    """

visits_data = await matomo.run_query(query_visits)

In [ ]:
visits_df = pd.DataFrame(visits_data, columns=columns)

In [ ]:
from urllib.parse import urlsplit, urlunsplit

def clean_url(url):
    if pd.isna(url) or not url:
        return url
    parts = urlsplit(url)
    # on ne garde que scheme + host + path, sans query ni fragment
    cleaned = urlunsplit((parts.scheme, parts.netloc, parts.path, "", ""))
    # slash final (mais pas la racine "https://site/")
    if cleaned.endswith("/") and parts.path != "/":
        cleaned = cleaned[:-1]
    return cleaned

visits_df["action_url_clean"] = visits_df["action_url"].apply(clean_url)

## Taux de non interaction : x% des usagers arrivés via moteur de recherche sur une page CC personnalisée n'interagissent pas avec la modale et continuent de lire la réponse pré affichée sans savoir si c'est la leur.

In [ ]:
df = visits_df.copy()

In [ ]:
TARGET_PATH = "https://code.travail.gouv.fr/contribution/1351-quelle-est-la-duree-du-preavis-en-cas-de-demission"

In [ ]:
df["actions"] = pd.to_numeric(df["actions"], errors="coerce")

# Page views uniquement, ordonnées dans la visite
pv = (
    df[df["action_type"] == "action"]
    .assign(action_timestamp=lambda x: pd.to_datetime(x["action_timestamp"], utc=True))
    .sort_values(["idvisit", "action_timestamp", "actions"], kind="stable")
)

# Première page vue de chaque visite
first_pv = pv.groupby("idvisit", as_index=False).first()[["idvisit", "action_url_clean"]]

# Visites dont la 1re page est la page cible
target_visits = first_pv.loc[first_pv["action_url_clean"] == TARGET_PATH, "idvisit"]

# Toutes les lignes (page_views + events + downloads) de ces visites, regroupées
df_target = (
    df[df["idvisit"].isin(target_visits)]
    .sort_values(["idvisit", "actions"])
)
visits_grouped = df_target.groupby("idvisit")

print(f"{len(target_visits)} visites commencent par la page cible")

In [ ]:
# Toutes les visites ayant vu la page cible (à n'importe quelle position)
all_visits = pv.loc[pv["action_url_clean"] == TARGET_PATH, "idvisit"].unique()

nb_all = len(all_visits)
nb_first = len(target_visits)
pct_first = nb_first / nb_all * 100 if nb_all else 0

print(f"Visites ayant consulté la page : {nb_all}")
print(f"  dont en 1re page (point d'entrée) : {nb_first} ({pct_first:.1f} %)")
print(f"  dont arrivées après navigation    : {nb_all - nb_first} ({100 - pct_first:.1f} %)")

In [ ]:
first_pv_target = (
    pv[pv["idvisit"].isin(target_visits)]
    .groupby("idvisit")
    .head(1)
)

nb_first = len(first_pv_target)

# --- Par type de source (direct, search, website, campaign, social) ---
by_type = first_pv_target["referrertype"].fillna("(vide)").value_counts().rename("visites").to_frame()
by_type["%"] = (by_type["visites"] / nb_first * 100).round(1)
print(by_type)

# --- Par source détaillée (Google, Bing, site référent, nom de campagne...) ---
by_name = (
    first_pv_target
    .assign(referrername=first_pv_target["referrername"].fillna("(vide)"))
    .groupby(["referrertype", "referrername"], dropna=False)
    .size()
    .rename("visites")
    .reset_index()
    .sort_values("visites", ascending=False)
)
by_name["%"] = (by_name["visites"] / nb_first * 100).round(1)
print(by_name.head(25).to_string(index=False))

In [ ]:
# Restreint aux visites dont la page cible est la 1re page
pv_target = pv[pv["idvisit"].isin(target_visits)]

stats = (
    pv_target.groupby("idvisit")
    .agg(
        nb_pages=("action_id", "size"),
        visitduration=("visitduration", "first"),
    )
    .reset_index()
)
stats["visitduration"] = pd.to_numeric(stats["visitduration"], errors="coerce")

# --- Répartition par nombre de pages vues ---
bins_pages = [0, 1, 2, 3, 5, 10, float("inf")]
labels_pages = ["1 page", "2 pages", "3 pages", "4-5 pages", "6-10 pages", "11+ pages"]
stats["pages_bucket"] = pd.cut(stats["nb_pages"], bins=bins_pages, labels=labels_pages)

repart_pages = (
    stats["pages_bucket"].value_counts(sort=False)
    .rename("visites").to_frame()
)
repart_pages["%"] = (repart_pages["visites"] / len(stats) * 100).round(1)
print(repart_pages)
print(f"\nMédiane : {stats['nb_pages'].median():.0f} pages | Moyenne : {stats['nb_pages'].mean():.2f}")

# --- Répartition par durée de visite ---
bins_dur = [-1, 0, 30, 60, 180, 600, float("inf")]
labels_dur = ["0 s", "1-30 s", "31-60 s", "1-3 min", "3-10 min", "10+ min"]
stats["duration_bucket"] = pd.cut(stats["visitduration"], bins=bins_dur, labels=labels_dur)

repart_dur = (
    stats["duration_bucket"].value_counts(sort=False)
    .rename("visites").to_frame()
)
repart_dur["%"] = (repart_dur["visites"] / len(stats) * 100).round(1)
print(repart_dur)
print(f"\nMédiane : {stats['visitduration'].median():.0f} s | Moyenne : {stats['visitduration'].mean():.0f} s")

In [ ]:
MOBILE_OS = {"iOS", "Android", "iPadOS", "HarmonyOS", "KaiOS", "Windows Phone", "Windows Mobile", "Tizen"}

def device_type(os_name):
    if not isinstance(os_name, str) or not os_name:
        return "inconnu"
    return "mobile" if os_name in MOBILE_OS else "desktop"

stats = (
    pv_target.groupby("idvisit")
    .agg(
        nb_pages=("action_id", "size"),
        visitduration=("visitduration", "first"),
        os=("operatingsystemname", "first"),
    )
    .reset_index()
)
stats["visitduration"] = pd.to_numeric(stats["visitduration"], errors="coerce")
stats["device"] = stats["os"].map(device_type)
stats["pages_bucket"] = pd.cut(stats["nb_pages"], bins=bins_pages, labels=labels_pages)
stats["duration_bucket"] = pd.cut(stats["visitduration"], bins=bins_dur, labels=labels_dur)

print(stats["device"].value_counts().to_frame("visites").assign(**{"%": lambda d: (d["visites"] / len(stats) * 100).round(1)}))

def repart_by_device(col):
    ct = pd.crosstab(stats[col], stats["device"])
    pct = (ct / ct.sum() * 100).round(1)
    out = pd.concat({"visites": ct, "%": pct}, axis=1)
    return out.swaplevel(axis=1).sort_index(axis=1, level=0)

print("\n=== Nombre de pages vues ===")
print(repart_by_device("pages_bucket"))
print(stats.groupby("device")["nb_pages"].agg(médiane="median", moyenne="mean").round(2))

print("\n=== Durée de visite ===")
print(repart_by_device("duration_bucket"))
print(stats.groupby("device")["visitduration"].agg(médiane="median", moyenne="mean").round(0))

In [ ]:
# Libellé = catégorie / action (le name est ignoré pour regrouper les variantes)
on_first_page = on_first_page.assign(
    event_label=on_first_page["action_eventcategory"].fillna("")
    + " / " + on_first_page["action_eventaction"].fillna("")
)

visit_bucket = pd.cut(nb_events_per_visit, bins=bins_ev, labels=labels_ev).rename("bucket")

# Pour chaque tranche : nb et % de visites de la tranche où l'event apparaît
ev_by_bucket = (
    on_first_page[["idvisit", "event_label"]]
    .drop_duplicates()
    .merge(visit_bucket, left_on="idvisit", right_index=True)
    .groupby(["bucket", "event_label"], observed=True)
    .size()
    .rename("nb_visites")
    .reset_index()
)
ev_by_bucket["% tranche"] = (
    ev_by_bucket["nb_visites"] / ev_by_bucket["bucket"].map(repart_ev["visites"]).astype(float) * 100
).round(1)
ev_by_bucket = ev_by_bucket.sort_values(["bucket", "nb_visites"], ascending=[True, False])

# Tableau récap : une ligne par tranche, liste "cat / action (xx %)" 
repart_ev["events vus"] = (
    ev_by_bucket.groupby("bucket", observed=True)
    .apply(lambda g: " | ".join(f"{l} ({p} %)" for l, p in zip(g["event_label"], g["% tranche"])))
)
pd.set_option("display.max_colwidth", None)
# print(repart_ev)

# Vue détaillée (plus lisible si beaucoup d'events)
print(ev_by_bucket.to_string(index=False))

## On garde que les informations sur la visite de la contribution

In [ ]:
# Tri fin : timestamp d'abord, actions en départage
df_first_page = (
    df_target
    .assign(action_timestamp=pd.to_datetime(df_target["action_timestamp"], utc=True))
    .sort_values(["idvisit", "action_timestamp", "actions"], kind="stable")
    .copy()
)

# Nombre de page views vus jusqu'à cette ligne incluse, dans la visite
df_first_page["pv_cum"] = (
    (df_first_page["action_type"] == "action").astype(int)
    .groupby(df_first_page["idvisit"]).cumsum()
)

# On garde tout jusqu'au 1er page view inclus, et on coupe dès le 2e
df_first_page = (
    df_first_page[df_first_page["pv_cum"] <= 1]
    .drop(columns="pv_cum")
    .reset_index(drop=True)
)

print(f"{df_first_page['idvisit'].nunique()} visites, {len(df_first_page)} lignes")
print(df_first_page["action_type"].value_counts())

# Contrôle : 1 seul page view par visite
assert (df_first_page[df_first_page["action_type"] == "action"].groupby("idvisit").size() == 1).all()

In [ ]:
# Premier event de chaque visite (df_first_page est déjà trié par idvisit, timestamp, actions)
first_event = (
    df_first_page[df_first_page["action_type"] == "event"]
    .groupby("idvisit")
    .head(1)
)

nb_visites = df_first_page["idvisit"].nunique()

first_event_summary = (
    first_event
    .groupby(["action_eventcategory", "action_eventaction"], dropna=False)
    .size()
    .rename("nb_visites")
    .reset_index()
    .sort_values("nb_visites", ascending=False)
)
first_event_summary["% visites"] = (first_event_summary["nb_visites"] / nb_visites * 100).round(1)

print(f"Visites sans aucun event sur la première page : {nb_visites - len(first_event)} "
      f"({(nb_visites - len(first_event)) / nb_visites * 100:.1f} %)\n")
print(first_event_summary.to_string(index=False))

In [ ]:
AFFICHER_ACTIONS = [
    "click_afficher_les_informations_CC",
    "click_afficher_les_informations_sans_CC",
    "click_afficher_les_informations_générales",
]

events_fp = df_first_page[df_first_page["action_type"] == "event"]
mask_afficher = events_fp["action_eventaction"].isin(AFFICHER_ACTIONS)

nb_visites = df_first_page["idvisit"].nunique()

# Global : au moins un des trois events dans la visite
visits_afficher = events_fp.loc[mask_afficher, "idvisit"].nunique()
print(f"Visites avec un clic 'afficher les informations' : {visits_afficher} / {nb_visites} "
      f"({visits_afficher / nb_visites * 100:.1f} %)\n")

# Détail par action (une visite peut en avoir plusieurs, la somme peut dépasser le global)
detail = (
    events_fp[mask_afficher]
    .groupby("action_eventaction")["idvisit"].nunique()
    .reindex(AFFICHER_ACTIONS, fill_value=0)
    .rename("nb_visites").to_frame()
)
detail["% visites"] = (detail["nb_visites"] / nb_visites * 100).round(1)
print(detail)